In [13]:
import pandas as pd
import time
import requests
import zipfile
import os
import json
import glob
import csv

# 1.Define URL BASE DATA

In [14]:

base_url = "https://opentender.net/api/tender/export-ocds-batch/"
years = list(range(2015, 2025))

all_data = []
years

[2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

# 2. fetching DATA

In [16]:
# List target ID yang diminta
target_lspe = {
    "127": "Jakarta",
    "15": "Jawa Timur",
    "42": "Jawa Tengah"
}

for lspe_id, nama_daerah in target_lspe.items():
    print(f"\n=== Memulai Download untuk {nama_daerah} (ID: {lspe_id}) ===")
    extract_folder = f"../data/json/{lspe_id}"
    os.makedirs(extract_folder, exist_ok=True)

    for year in years:
        target_url = f"{base_url}?year={year}&lpse={lspe_id}"
        print(f"Downloading: Year {year}...")
        try:
            res = requests.get(target_url, stream=True)
            zip_filename = f"ocds_{lspe_id}_{year}.zip"
            
            with open(zip_filename, "wb") as f:
                for chunk in res.iter_content(chunk_size=8192):
                    if chunk: f.write(chunk)
            
            with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
                for member in zip_ref.namelist():
                    if member.endswith(".json"):
                        zip_ref.extract(member, extract_folder)
                        filename = os.path.basename(member)
                        old_path = os.path.join(extract_folder, filename)
                        new_name = f"ocds-data-tender-{year}-{lspe_id}.json"
                        new_path = os.path.join(extract_folder, new_name)
                        if os.path.exists(new_path): os.remove(new_path)
                        os.rename(old_path, new_path)
            
            os.remove(zip_filename)
            time.sleep(1) # Jeda sebentar agar tidak membebani server
            
        except Exception as e:
            print(f"Error for {year} in {nama_daerah}: {e}")


=== Memulai Download untuk Jakarta (ID: 127) ===
Downloading: Year 2015...
Downloading: Year 2016...
Downloading: Year 2017...
Downloading: Year 2018...
Downloading: Year 2019...
Downloading: Year 2020...
Downloading: Year 2021...
Downloading: Year 2022...
Downloading: Year 2023...
Downloading: Year 2024...
Error for 2024 in Jakarta: File is not a zip file

=== Memulai Download untuk Jawa Timur (ID: 15) ===
Downloading: Year 2015...
Downloading: Year 2016...
Downloading: Year 2017...
Downloading: Year 2018...
Downloading: Year 2019...
Downloading: Year 2020...
Downloading: Year 2021...
Downloading: Year 2022...
Downloading: Year 2023...
Downloading: Year 2024...

=== Memulai Download untuk Jawa Tengah (ID: 42) ===
Downloading: Year 2015...
Downloading: Year 2016...
Downloading: Year 2017...
Downloading: Year 2018...
Downloading: Year 2019...
Downloading: Year 2020...
Downloading: Year 2021...
Downloading: Year 2022...
Downloading: Year 2023...
Downloading: Year 2024...
Error for 2024 

# 3.Convert json to CSV file

In [18]:
# define path folder & output path
path_folder = f'../data/json/{lspe_id}'
output_file_master = '../data/csv/master_data_ocds.csv'
os.makedirs('../data/csv', exist_ok=True)

fieldnames = [
    'lspe_id', 'nama_daerah', 'ocid', 'release_id', 'date', 'buyer_name', 'buyer_id',
    'tender_id', 'tender_title', 'mainProcurementCategory',
    'tender_minValue', 'tender_datePublished', 'tender_status',
    'award_id', 'award_title', 'award_date', 'award_value', 'award_supplier'
]

total_rows_global = 0

# Buka file Master sekali saja di awal
with open(output_file_master, 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

    # Loop melalui setiap daerah
    for lspe_id, nama_daerah in target_lspe.items():
        path_folder = f'../data/json/{lspe_id}'
        
        # Cari file ocds .json untuk daerah tersebut
        files = glob.glob(os.path.join(path_folder, f"ocds-data-tender-*-{lspe_id}.json"))
        
        if not files:
            print(f"Warning: Folder {lspe_id} ({nama_daerah}) kosong!")
            continue

        print(f"Processing {nama_daerah} (ID: {lspe_id})...")
        rows_per_region = 0

        for file_path in files:
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)

                for release in data.get('releases', []):
                    tender = release.get('tender', {})
                    buyer = release.get('buyer', {})
                    awards = release.get('awards', [])

                    for award in awards:
                        suppliers = award.get('suppliers', [])
                        supplier_names = ', '.join([s.get('name', '') for s in suppliers]) or None

                        # Tulis ke Master CSV dengan tambahan kolom daerah
                        writer.writerow({
                            'lspe_id': lspe_id,
                            'nama_daerah': nama_daerah,
                            'ocid': release.get('ocid'),
                            'release_id': release.get('id'),
                            'date': release.get('date'),
                            'buyer_name': buyer.get('name'),
                            'buyer_id': buyer.get('id'),
                            'tender_id': tender.get('id'),
                            'tender_title': tender.get('title'),
                            'mainProcurementCategory': tender.get('mainProcurementCategory'),
                            'tender_minValue': tender.get('minValue', {}).get('amount'),
                            'tender_datePublished': tender.get('datePublished'),
                            'tender_status': tender.get('status'),
                            'award_id': award.get('id'),
                            'award_title': award.get('title'),
                            'award_date': award.get('date'),
                            'award_value': award.get('value', {}).get('amount'),
                            'award_supplier': supplier_names
                        })
                        rows_per_region += 1
                        total_rows_global += 1

            except Exception as e:
                print(f"Gagal membaca {file_path}: {e}")
        
        print(f"Berhasil memproses {rows_per_region} baris dari {nama_daerah}.")

print(f"\n--- SELESAI ---")
print(f"Total data gabungan: {total_rows_global} baris.")
print(f"File disimpan di: {output_file_master}")

Processing Jakarta (ID: 127)...
Berhasil memproses 8678 baris dari Jakarta.
Processing Jawa Timur (ID: 15)...
Berhasil memproses 6440 baris dari Jawa Timur.
Processing Jawa Tengah (ID: 42)...
Berhasil memproses 6038 baris dari Jawa Tengah.

--- SELESAI ---
Total data gabungan: 21156 baris.
File disimpan di: ../data/csv/master_data_ocds.csv
